<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/FinalProject/FinalProject_Overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY386 — Final Project 2026
## Topic Overview
**Instructor:** Tim Thomay · **Date:** 2026-04-23

This notebook is the menu of suggested final-project topics. You may **propose your own topic** instead — just clear it with me first.

### Theme: Machine Learning in Physics
Building directly on **HW5** (Decision Tree, k-Nearest Neighbors, Random Forest) and **HW6** (Multi-Layer Perceptron — sklearn's `MLPClassifier` — on SDSS data via `astroquery`), you'll pick a dataset, frame a physics question as a machine-learning problem, run the full pipeline, and report what you found.

### Ground rules
- **Submission:** Fork → `yourname-final` branch → pull request into `Homework2026`, reviewer `@laserlab`, milestone `Final-2026`.
- **Deliverable:** one `.ipynb` in your student folder, about 42 points worth of tasks, publication-quality plots, NumPy-style docstrings + type annotations.
- **Due:** finals week.
- **Teams:** solo, unless the scope justifies a pair. Talk to me.
- **AI disclosure** required in the pull-request description, same as HW4–HW6.

---

## At a glance

| # | Topic | Data source | Machine-learning type | Main tools |
|---|-------|-------------|-----------------------|------------|
| 1 | Star-color classification from Seestar images | Our own 2025 FITS files | Classification | `astropy`, `photutils`, `astroquery.gaia`, scikit-learn (Random Forest, k-Nearest Neighbors) |
| 2 | Ring of Fire from earthquake catalog | USGS `libcomcat` | Classification / clustering | `libcomcat`, scikit-learn (k-Nearest Neighbors, Random Forest), `cartopy` |
| 3 | Atmospheric methane (CH₄) forecast | NOAA GML public CSV | **Regression / time series** (stretch) | `pandas`, `scipy.optimize.curve_fit`, `statsmodels` |
| 4 | Exoplanet confirmation classifier | NASA Exoplanet Archive via `astroquery` | Classification | `astroquery`, scikit-learn (Decision Tree, Random Forest, Multi-Layer Perceptron) |
| 5 | Gaia variable-star classification | Gaia DR3 via `astroquery.gaia` | Multi-class classification | `astroquery`, scikit-learn (Multi-Layer Perceptron, Random Forest) |

All five topics reuse the HW5/HW6 workflow: load → clean → scale → fit → confusion matrix → feature importance → physical interpretation. Topic 3 is the regression stretch — you'd swap classification metrics for root-mean-square error and prediction intervals.

---

## Topic 1 — Seestar Star-Color Classification

**The question.** Can we predict a star's spectral class (O/B/A/F/G/K/M) purely from color indices measured in our own Seestar S30 images?

**The data.** Stacked FITS files I took last year with the Seestar. Good candidates:
- **M45 Pleiades** — open cluster, young (~100 Myr), lots of hot B-type stars.
- **M44 Praesepe / Beehive** — open cluster, middle-aged.
- **M5** — globular cluster, old (~13 Gyr), dominated by red giants.

The contrast between Pleiades and M5 is the punchline: same instrument, very different stellar populations.

**The pipeline.**
1. Load the stacked FITS with `astropy.io.fits`.
2. Detect stars with `photutils.DAOStarFinder`; do aperture photometry on each color channel.
3. Compute a color index from channel ratios (an approximate B–V proxy).
4. Cross-match the brightest N stars with **Gaia DR3** via `astroquery.gaia` to pull their real BP–RP colors and spectral labels.
5. Train Random Forest and k-Nearest Neighbors classifiers on the labeled subset, then predict for the rest.

**Stretch goal.** Plot a color–magnitude diagram and overlay the classifier's decision regions. Compare Pleiades vs. M5 — the main sequence looks different because the populations *are* different.

**Physics connection.** Hertzsprung–Russell diagram, blackbody spectra, stellar evolution.

---

## Topic 2 — Ring of Fire from Earthquake Data

**The question.** Can a machine learning model rediscover the Ring of Fire purely from earthquake locations and depths? Which features matter most — position, depth, or magnitude?

**The data.** USGS `libcomcat` — official Python client for the USGS ComCat earthquake archive. Think of it as `astroquery` for geophysics.

```python
from libcomcat.search import search
from datetime import datetime
events = search(starttime=datetime(2020,1,1),
                endtime=datetime(2024,12,31),
                minmagnitude=5.0)
```

That pulls several thousand M5+ earthquakes worldwide — lat, lon, depth, magnitude.

**The pipeline.**
1. Query USGS → pandas DataFrame.
2. Label each event as "Ring of Fire" member or not (simple polygon around the Pacific rim, or use a published tectonic-boundary shapefile).
3. **KNN** on (lat, lon) — does geography alone nail it?
4. **Random Forest** on (lat, lon, depth, magnitude) — does depth help? (Yes: subduction zones are deep.)
5. Compare; inspect feature importances.

**Stretch goal.** `cartopy` world map with colored decision regions overlaid on actual plate boundaries.

**Physics connection.** Plate tectonics — subduction zones produce deep, high-magnitude quakes; mid-ocean ridges are shallow. You should be able to read subduction off the classifier.

---

## Topic 3 — Atmospheric Methane (CH₄) Forecast  *(regression stretch)*

**The question.** What will globally-averaged atmospheric methane be in 2030? And how confident should we be?

**Why this is the "stretch" topic.** HW5/HW6 only covered **classification**. This one is **regression on a time series**, so you'll learn something genuinely new on top of the HW5/HW6 toolkit.

**The data.** NOAA GML globally-averaged marine-surface CH₄ — one plain-text URL, monthly means from July 1983 to now:

```python
import pandas as pd
url = "https://gml.noaa.gov/webdata/ccgg/trends/ch4/ch4_mm_gl.txt"
df = pd.read_csv(url, comment="#", sep=r"\s+",
                 names=["year","month","decimal",
                        "average","average_unc","trend","trend_unc"])
```

Values are in parts per billion (ppb). `-9.9` means the uncertainty has not yet been computed for that month — treat as missing.

**The pipeline.**
1. Load the monthly time series.
2. Decompose into trend + seasonal + residual with `statsmodels.tsa.seasonal_decompose`.
3. Fit the trend with `scipy.optimize.curve_fit` (linear, then quadratic — compare residuals).
4. Fit the seasonal component with Fourier terms.
5. Extrapolate to 2030 with a confidence band.
6. Optional: sklearn's `MLPRegressor` (a Multi-Layer Perceptron used for regression) on lagged features, as a second estimate.

**Why methane is more interesting than CO₂.** The CH₄ trend *plateaued 1999–2006* (biospheric sink balance), then **re-accelerated after ~2007**, with another jump around 2020. A single polynomial fit *will fail* — you'll need a piecewise or change-point model. This is the whole point of the project.

**Stretch goal.** Cross-validate on the last 5 years held out. Compare pre- vs. post-2007 fits. Load CO₂ from `statsmodels.datasets.co2` as a second signal — which gas is easier to forecast, and why?

**Physics connection.** Greenhouse-gas atmospheric budget, biospheric carbon cycle, model-selection bias.

---

## Topic 4 — Exoplanet Confirmation Classifier

**The question.** Given a Kepler transit signal, can we predict whether it's a *real exoplanet* or a *false positive* (eclipsing binary, instrumental artifact)? Which features tell you?

**The data.** NASA Exoplanet Archive via `astroquery` — same library you used in HW6, pointed at a different archive:

```python
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
tab = NasaExoplanetArchive.query_criteria(
    table="cumulative",
    select=("kepid, koi_period, koi_duration, koi_depth, koi_prad, "
            "koi_steff, koi_slogg, koi_srad, koi_disposition"))
df = tab.to_pandas()
```

About 10,000 Kepler Objects of Interest, each with transit parameters, stellar parameters, and a label (`CONFIRMED` / `CANDIDATE` / `FALSE POSITIVE`).

**The pipeline.**
1. Query with astroquery → pandas → missing-value handling.
2. Train Decision Tree, Random Forest, and Multi-Layer Perceptron classifiers on CONFIRMED vs. FALSE POSITIVE.
3. Look at feature importances — what makes a signal "real"?
4. Apply the best model to the CANDIDATE subset and rank by predicted-planet probability.

**Stretch goal.** Re-query the **TOI** (TESS Objects of Interest) table by changing one argument. Does the classifier generalize across missions?

**Physics connection.** Transit photometry, stellar binary contamination, how machine learning is actually used in the Kepler vetting pipeline.

---

## Topic 5 — Gaia Variable-Star Classification

**The question.** Can we classify variable stars into their physical types (RR Lyrae, Cepheid, eclipsing binary, δ Scuti, long-period variable, …) from Gaia photometry and astrometry?

**The data.** Gaia DR3 via `astroquery.gaia`, using an ADQL query that joins the variability-classifier table with the main Gaia source table:

```python
from astroquery.gaia import Gaia
job = Gaia.launch_job_async("""
    SELECT TOP 20000
      v.source_id, v.best_class_name, v.best_class_score,
      g.phot_g_mean_mag, g.bp_rp, g.parallax, g.pmra, g.pmdec,
      g.phot_variable_flag
    FROM gaiadr3.vari_classifier_result AS v
    JOIN gaiadr3.gaia_source AS g USING (source_id)
    WHERE v.best_class_score > 0.9
""")
df = job.get_results().to_pandas()
```

**The pipeline.**
1. Run the ADQL query via astroquery → pandas.
2. `StandardScaler` + the four classifiers from HW5 / HW6 (Decision Tree, k-Nearest Neighbors, Random Forest, Multi-Layer Perceptron).
3. Multi-class confusion matrix across variability types.
4. Feature importances: what separates RR Lyrae from Cepheids? (Hint: period and metallicity proxies.)

**Stretch goal.** Plot a period–luminosity diagram colored by predicted class. Overlay the classical Leavitt law for Cepheids. Does the machine-learning classifier recover known physics?

**Physics connection.** Stellar pulsation mechanisms, eclipsing geometry, the period–luminosity relation as the cosmic distance ladder's second rung.